# LMS Adaptive Equalizer | Interactive Project Notebook
### Modern Signal Processing · BPSK and QPSK

**Main question:** Can an adaptive LMS FIR filter reduce intersymbol interference (ISI) and recover digital symbols more accurately?

**Project workflow:** `Bits → Modulation → Dispersive channel + AWGN → LMS equalizer → Decisions → BER / EVM / MSE`

> **Scope:** Symbol-spaced, synchronized simulation; no physical transmission, pulse shaping, or timing recovery. Training uses known symbols; evaluation uses unseen test symbols with frozen weights.

**Run instructions:** Open this notebook from the **project root folder** and choose **Kernel → Restart & Run All**. The helper files `lms_equalizer.py` and `experiments.py` must remain beside the notebook.

---
### Contents
1. Communication problem and objectives
2. LMS mathematical model
3. Imports and experiment settings
4. Modulation sanity check
5. BPSK simulation and metrics
6. QPSK simulation and metrics
7. Reproduce all experiments
8. Visual analysis (nine figures)
9. Conclusions, limitations, and viva preparation

## 1. Communication problem and objectives

A dispersive channel spreads each transmitted symbol into neighboring symbol periods, causing **intersymbol interference (ISI)**. Noise makes detection harder.

| Stage | Input | Operation | Output |
|---|---|---|---|
| Transmitter | Random bits | BPSK or QPSK mapping | Symbols $s[n]$ |
| Channel | $s[n]$ | FIR multipath + AWGN | Distorted samples $r[n]$ |
| Equalizer | $r[n]$ | LMS coefficient adaptation | Estimates $y[n]$ |
| Receiver | $y[n]$ | Hard decision | Estimated bits |
| Evaluation | Original and estimated bits | BER / EVM / MSE | Performance comparison |

$$r[n]=\sum_{k=0}^{L_h-1}h[k]s[n-k]+v[n]$$

**We will test:** (i) BER before and after LMS, (ii) convergence, (iii) step-size and filter-length sensitivity, (iv) SNR and channel effects.

## 2. Mathematical foundation: LMS in matrix form

For $M$ equalizer taps:

$$\mathbf{x}[n]=[r[n],r[n-1],\ldots,r[n-M+1]]^T$$

$$y[n]=\mathbf{w}^H[n]\mathbf{x}[n],\qquad d[n]=s[n-D]$$

$$e[n]=d[n]-y[n]$$

**Core update rule:**

$$\boxed{\mathbf{w}[n+1]=\mathbf{w}[n]+\mu\mathbf{x}[n]e^*[n]}$$

Here $\mu$ is the step size, $D$ is decision delay, and $^*$ denotes complex conjugation (essential for QPSK).

The mean-square objective is $J(\mathbf w)=E[|e[n]|^2]$. The Wiener reference solution satisfies $\mathbf R\mathbf w_{opt}=\mathbf p$ when $\mathbf R$ is invertible. LMS estimates a useful solution incrementally, without explicitly inverting $\mathbf R$.

**Important:** The common bound $0<\mu<2/\lambda_{max}(\mathbf R)$ is a mean-convergence guide under idealized assumptions, not a universal stability guarantee.

## 3. Environment and reproducible settings

The settings below are the **same as the original project**. Changing them affects the two example simulations below; the full sweep uses the settings inside `experiments.py`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
from lms_equalizer import modulate, demodulate, lms_equalizer, simulate, CHANNELS
from experiments import generate

SEED = 2026
MU = 0.018
NUM_TAPS = 9
DELAY = 2
TRAIN_SYMBOLS = 2500
TEST_SYMBOLS = 9000
SNR_DB = 12.0
print("Channel impulse response:", CHANNELS["mild"])
print("Configuration:", MU, "step size;", NUM_TAPS, "taps;", DELAY, "decision delay")

### Configuration guide

| Parameter | Meaning | Current value |
|---|---|---|
| `SEED` | Reproducible random stream | 2026 |
| `MU` | LMS learning step size | 0.018 |
| `NUM_TAPS` | Number of equalizer FIR coefficients | 9 |
| `DELAY` | Target symbol alignment | 2 |
| `TRAIN_SYMBOLS` | Known symbols used for learning | 2,500 |
| `TEST_SYMBOLS` | Unseen symbols used for scoring | 9,000 |
| `SNR_DB` | Channel sample-power SNR | 12 dB |

**Note:** This SNR definition is not identical to $E_b/N_0$.

## 4. Modulation sanity check

Before channel simulation, verify that BPSK and QPSK symbol mapping and demapping are consistent. This is a simple test, **not** a channel-performance result.

In [ ]:
bits = np.array([0,0, 0,1, 1,0, 1,1], dtype=int)
for scheme in ("BPSK", "QPSK"):
    symbols = modulate(bits, scheme)
    assert np.array_equal(demodulate(symbols, scheme), bits)
    print(scheme, ":", symbols, " | unit power:", np.mean(np.abs(symbols)**2))

## 5. Experiment A — BPSK equalization

**Goal:** Compare the received signal before equalization against the output of an LMS equalizer on an independent test segment.

**Expected reading:** Lower **BER** is better. Lower **EVM** is better. Weights are learned using the training sequence and then frozen for testing.

In [ ]:
bpsk = simulate("BPSK", snr_db=SNR_DB, mu=MU, num_taps=NUM_TAPS,
                delay=DELAY, n_train=TRAIN_SYMBOLS, n_test=TEST_SYMBOLS,
                seed=SEED, track_weights=True)
print(f"BPSK raw BER: {bpsk.raw_ber:.5f}; equalized BER: {bpsk.equalized_ber:.5f}")
print(f"BPSK raw EVM: {bpsk.raw_evm:.3f}; equalized EVM: {bpsk.equalized_evm:.3f}")
print("Final learned weights:", np.round(bpsk.lms.weights, 3))
assert len(bpsk.test_indices) == TEST_SYMBOLS

## 6. Experiment B — QPSK equalization

QPSK has complex-valued symbols. LMS therefore uses the conjugated error in the coefficient update. Compare the same before/after metrics.

In [ ]:
qpsk = simulate("QPSK", snr_db=SNR_DB, mu=MU, num_taps=NUM_TAPS,
                delay=DELAY, n_train=TRAIN_SYMBOLS, n_test=TEST_SYMBOLS,
                seed=SEED, track_weights=True)
print(f"QPSK raw BER: {qpsk.raw_ber:.5f}; equalized BER: {qpsk.equalized_ber:.5f}")
print(f"QPSK raw EVM: {qpsk.raw_evm:.3f}; equalized EVM: {qpsk.equalized_evm:.3f}")
print("Complex weights:", np.round(qpsk.lms.weights, 3))
assert np.iscomplexobj(qpsk.lms.weights)

### Quick comparison dashboard

The table below is generated from the actual current BPSK and QPSK simulation objects (not manually entered numbers).

In [ ]:
print(f"{'Modulation':<12} {'Raw BER':>12} {'LMS BER':>12} {'Raw EVM':>12} {'LMS EVM':>12}")
print('-' * 64)
for name, result in [('BPSK', bpsk), ('QPSK', qpsk)]:
    print(f'{name:<12} {result.raw_ber:>12.4%} {result.equalized_ber:>12.4%} {result.raw_evm:>12.3f} {result.equalized_evm:>12.3f}')
    if result.raw_ber > 0:
        reduction = 100 * (result.raw_ber - result.equalized_ber) / result.raw_ber
        print(f'  Relative BER reduction: {reduction:.1f}%')

## 7. Full experiment suite

The original `experiments.generate()` function regenerates all **nine graphs** plus machine-readable result files. This can take a little time. Run this cell before the figure sections.

In [ ]:
summary = generate()  # Generates and saves the exact plots used in the report/presentation.
figure_dir = Path("results/simulation_graphs")
print("Main experiment summary:")
for name, result in summary["main_experiments"].items():
    print(f"  {name}: BER {result['raw_BER']:.4f} -> {result['LMS_BER']:.4f}")

## 8. Visual results and interpretation

All nine plots below come from the **existing simulation code**, with no changes to its algorithms or experiment definitions. Each graph is immediately followed by its interpretation guide.

### 8.1 — LMS training convergence

**What to look for:** MSE should generally decrease during adaptation; the final error need not be zero.

In [ ]:
display(Image(filename=str(figure_dir / "01_mse_convergence.png"), width=850))

### 8.2 — QPSK constellation

**What to look for:** Compare the distorted cloud with the equalized symbols and ideal constellation locations.

In [ ]:
display(Image(filename=str(figure_dir / "02_qpsk_constellation.png"), width=850))

### 8.3 — BPSK signal comparison

**What to look for:** Look at how the channel and equalizer affect the symbol-spaced waveform.

In [ ]:
display(Image(filename=str(figure_dir / "03_bpsk_waveforms.png"), width=850))

### 8.4 — Step-size sensitivity

**What to look for:** Small step sizes adapt slowly; large step sizes can increase residual error or cause instability.

In [ ]:
display(Image(filename=str(figure_dir / "04_stepsize_sweep.png"), width=850))

### 8.5 — Equalizer length

**What to look for:** More taps offer flexibility but can also slow learning and enhance noise.

In [ ]:
display(Image(filename=str(figure_dir / "05_filter_length_sweep.png"), width=850))

### 8.6 — BER versus SNR

**What to look for:** Lower BER is better; compare raw and LMS results across noise conditions.

In [ ]:
display(Image(filename=str(figure_dir / "06_ber_vs_snr.png"), width=850))

### 8.7 — Effective channel response

**What to look for:** Inspect whether the channel–equalizer cascade becomes more concentrated near one delay.

In [ ]:
display(Image(filename=str(figure_dir / "07_effective_impulse_response.png"), width=850))

### 8.8 — Adaptive tap evolution

**What to look for:** Tap trajectories show how the equalizer learns during training.

In [ ]:
display(Image(filename=str(figure_dir / "08_coefficient_trajectories.png"), width=850))

### 8.9 — Different channel conditions

**What to look for:** Compare performance under the channel cases defined by the original experiment code.

In [ ]:
display(Image(filename=str(figure_dir / "09_channel_comparison.png"), width=850))

## 9. Findings and conclusion

**Central conclusion:** LMS adaptation aims to undo part of the dispersive channel's ISI, improving the reliability of symbol decisions. Judge success from the actual generated BER/EVM and convergence plots, not from an assumed improvement.

**What this project demonstrates:**

1. A symbol-spaced BPSK/QPSK communication chain with FIR channel distortion and AWGN.
2. Supervised LMS adaptation on a known training sequence.
3. Evaluation on held-out symbols after freezing the learned coefficients.
4. Parameter sensitivity to step size, tap count, SNR, and channel response.

**Limitations:** Ideal symbol timing, fixed channel models, no carrier-offset correction, and no oversampled eye diagram. These are simulated results, not over-the-air measurements.

### Viva: five questions to prepare

1. **Why an equalizer?** To reduce ISI from a dispersive channel.
2. **Why LMS?** It adapts coefficients using an inexpensive error-driven update.
3. **Why a training sequence?** It supplies known desired symbols during learning.
4. **Why freeze weights for testing?** To evaluate on unseen data without using the answer to adapt.
5. **Why conjugate the error for QPSK?** Complex LMS requires it for the chosen coefficient convention.

### Reproduce outside the notebook

```bash
pip install -r requirements.txt
python -m unittest discover -s tests -v
python experiments.py
```

**Submission note:** Review and explain the simulation yourself; keep the AI log accurate and follow the course's repository/report requirements. See `README.md`, `docs/EXPERIMENT_GUIDE.md`, and `docs/VIVA_QA.md`.